# Bakery Business Intelligence
## Market Basket Analysis & Business Optimization

This project analyses transactional data from **The Bread Basket**, a bakery/café in Edinburgh, to identify actionable opportunities for a food retail business.

### Business questions

1. Which products drive the largest share of demand?
2. When does the bakery experience its highest transaction volume?
3. How large are customer baskets?
4. Which products are purchased together more often than expected by chance?
5. Where are the strongest opportunities for cross-selling, staffing and assortment optimization?

### Dataset

The public dataset contains bakery transactions recorded between late 2016 and early 2017.

Raw columns:

- `Date`
- `Time`
- `Transaction`
- `Item`

The raw version contains placeholder `NONE` records, which are removed during cleaning.

### Analytical workflow

**Raw transactions → Cleaning → EDA → Basket construction → Association rules → Business recommendations**

> Important limitation: the dataset does **not** contain prices, margins, costs, stock levels or waste. Therefore, this project identifies commercial and operational opportunities, but does not claim a direct profit uplift.


## 1. Imports

In [ ]:
from pathlib import Path
from itertools import combinations
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

## 2. Load the public dataset

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/prasertcbs/basic-dataset/master/BreadBasket_DMS.csv"

df_raw = pd.read_csv(DATA_URL)

print(f"Raw rows: {len(df_raw):,}")
display(df_raw.head())

## 3. Cleaning and feature engineering

In [ ]:
df = df_raw.copy()

# Standardize text and remove placeholder records
df["Item"] = df["Item"].astype(str).str.strip()
df = df[df["Item"].str.upper() != "NONE"].copy()

# Build timestamp
df["datetime"] = pd.to_datetime(
    df["Date"].astype(str) + " " + df["Time"].astype(str),
    errors="coerce",
)

df = df.dropna(subset=["datetime", "Transaction", "Item"])

df["date"] = df["datetime"].dt.date
df["hour"] = df["datetime"].dt.hour
df["weekday"] = df["datetime"].dt.day_name()
df["is_weekend"] = df["datetime"].dt.dayofweek >= 5

weekday_order = [
    "Monday", "Tuesday", "Wednesday", "Thursday",
    "Friday", "Saturday", "Sunday"
]

print(f"Valid line items: {len(df):,}")
print(f"Unique transactions: {df['Transaction'].nunique():,}")
print(f"Unique products: {df['Item'].nunique():,}")
print(f"Date range: {df['datetime'].min()} → {df['datetime'].max()}")

## 4. Product demand

In [ ]:
product_units = df["Item"].value_counts()
product_share = product_units / product_units.sum()

top_products = pd.DataFrame({
    "units": product_units,
    "share": product_share,
}).head(15)

display(top_products.style.format({"share": "{:.1%}"}))

In [ ]:
top10 = top_products.head(10).sort_values("units")

plt.figure(figsize=(9, 6))
plt.barh(top10.index, top10["units"])
plt.xlabel("Unidades registradas")
plt.ylabel("")
plt.title("Productos con mayor volumen de venta")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "01_top_products.png", dpi=200, bbox_inches="tight")
plt.show()

### Long-tail assortment

A product with low historical frequency is **not automatically unprofitable**.  
However, a very long tail of low-volume SKUs is a useful signal for an assortment review when combined with future data on margin, spoilage and storage requirements.

In [ ]:
low_frequency = (product_share < 0.01).sum()

print(
    f"{low_frequency} of {len(product_share)} products "
    f"represent individually less than 1% of recorded item volume."
)

## 5. Basket-size analysis

In [ ]:
# Unique products per transaction.
# For basket analysis, repeated units of the same product count once.
basket_size = (
    df.groupby("Transaction")["Item"]
      .nunique()
      .rename("basket_size")
)

single_item_share = (basket_size == 1).mean()

print(f"Average unique products per basket: {basket_size.mean():.2f}")
print(f"Median basket size: {basket_size.median():.0f}")
print(f"Single-item baskets: {single_item_share:.1%}")
print(f"Baskets with <= 5 unique products: {(basket_size <= 5).mean():.1%}")

In [ ]:
basket_dist = basket_size.value_counts().sort_index()

plt.figure(figsize=(8, 5))
plt.bar(basket_dist.index.astype(str), basket_dist.values)
plt.xlabel("Número de productos distintos por ticket")
plt.ylabel("Número de tickets")
plt.title("Distribución del tamaño de la cesta")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "02_basket_size.png", dpi=200, bbox_inches="tight")
plt.show()

## 6. Temporal demand

Two views are useful:

- **Transactions by hour** → workload and staffing.
- **Average transactions per operating day by weekday** → day-specific planning.

Transaction counts are used instead of raw line items so that large baskets do not artificially inflate customer-flow estimates.

In [ ]:
# One timestamp per ticket
tickets = (
    df.groupby("Transaction", as_index=False)
      .agg(datetime=("datetime", "min"))
)

tickets["date"] = tickets["datetime"].dt.date
tickets["hour"] = tickets["datetime"].dt.hour
tickets["weekday"] = tickets["datetime"].dt.day_name()

hourly_transactions = tickets.groupby("hour").size()

plt.figure(figsize=(9, 5))
plt.bar(hourly_transactions.index, hourly_transactions.values)
plt.xlabel("Hora del día")
plt.ylabel("Tickets")
plt.title("Distribución de transacciones por hora")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "03_transactions_by_hour.png", dpi=200, bbox_inches="tight")
plt.show()

peak_hour = hourly_transactions.idxmax()
print(f"Peak transaction hour: {peak_hour}:00")

In [ ]:
daily = (
    tickets.groupby(["date", "weekday"])
           .size()
           .rename("transactions")
           .reset_index()
)

weekday_avg = (
    daily.groupby("weekday")["transactions"]
         .mean()
         .reindex(weekday_order)
)

plt.figure(figsize=(9, 5))
plt.bar(weekday_avg.index, weekday_avg.values)
plt.xticks(rotation=35, ha="right")
plt.ylabel("Media de tickets por día")
plt.title("Demanda media por día de la semana")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "04_weekday_demand.png", dpi=200, bbox_inches="tight")
plt.show()

print("Highest average weekday:", weekday_avg.idxmax())
print("Lowest average weekday:", weekday_avg.idxmin())

## 7. Market Basket Analysis

Association-rule metrics:

- **Support**: proportion of transactions containing both products.
- **Confidence A → B**: probability of B being present when A is present.
- **Lift**: how much more often A and B appear together than expected if they were independent.

A lift above 1 indicates a positive association. This notebook focuses on rules with enough support to be commercially relevant rather than rare coincidences.

In [ ]:
# Convert transactions into sets of unique items
basket_series = (
    df.groupby("Transaction")["Item"]
      .apply(lambda x: frozenset(x))
)

n_transactions = len(basket_series)

item_transaction_count = Counter()
pair_count = Counter()

for basket in basket_series:
    for item in basket:
        item_transaction_count[item] += 1
    for pair in combinations(sorted(basket), 2):
        pair_count[pair] += 1

item_support = {
    item: count / n_transactions
    for item, count in item_transaction_count.items()
}

rules = []

for (a, b), count_ab in pair_count.items():
    support_ab = count_ab / n_transactions

    confidence_a_b = support_ab / item_support[a]
    confidence_b_a = support_ab / item_support[b]

    lift = support_ab / (item_support[a] * item_support[b])

    rules.append({
        "antecedent": a,
        "consequent": b,
        "support": support_ab,
        "confidence": confidence_a_b,
        "lift": lift,
        "co_occurrences": count_ab,
    })

    rules.append({
        "antecedent": b,
        "consequent": a,
        "support": support_ab,
        "confidence": confidence_b_a,
        "lift": lift,
        "co_occurrences": count_ab,
    })

rules_df = pd.DataFrame(rules)

# Keep rules with at least 1% support and positive association.
business_rules = (
    rules_df[
        (rules_df["support"] >= 0.01)
        & (rules_df["lift"] >= 1.20)
    ]
    .sort_values(["confidence", "lift"], ascending=False)
    .reset_index(drop=True)
)

display(
    business_rules.head(20).style.format({
        "support": "{:.2%}",
        "confidence": "{:.2%}",
        "lift": "{:.2f}",
    })
)

In [ ]:
top_rules = business_rules.head(10).copy()
top_rules["rule"] = top_rules["antecedent"] + " → " + top_rules["consequent"]
top_rules = top_rules.sort_values("confidence")

plt.figure(figsize=(10, 6))
plt.barh(top_rules["rule"], top_rules["confidence"])
plt.xlabel("Confidence")
plt.xlim(0, 1)
plt.title("Reglas de asociación con mayor confianza")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "05_association_rules.png", dpi=200, bbox_inches="tight")
plt.show()

## 8. Business opportunity diagnostics

In [ ]:
coffee_support = item_support.get("Coffee", np.nan)

peak_start = 9
peak_end = 14
peak_share = tickets["hour"].between(peak_start, peak_end).mean()

summary = pd.DataFrame({
    "Metric": [
        "Transactions",
        "Products",
        "Single-item baskets",
        "Coffee transaction support",
        f"Transactions between {peak_start}:00 and {peak_end}:59",
        "Low-frequency SKUs (<1% item volume)",
    ],
    "Value": [
        f"{n_transactions:,}",
        f"{df['Item'].nunique():,}",
        f"{single_item_share:.1%}",
        f"{coffee_support:.1%}",
        f"{peak_share:.1%}",
        f"{low_frequency}/{len(product_share)}",
    ],
})

display(summary)

## 9. Business conclusions

The code above recomputes the conclusions directly from the public dataset.

### 1. Cross-selling should focus on single-item baskets

A substantial share of transactions contains only one unique product. This creates a measurable **basket-expansion opportunity**.

**Business action:**  
Test contextual add-on prompts rather than broad discounts. For example, after a customer selects a beverage, recommend one complementary bakery item supported by association-rule evidence.

**How to validate:**  
A/B test conversion rate, average basket size and gross margin per transaction.

---

### 2. Coffee behaves as an anchor product

Coffee has very high transaction support and appears repeatedly in strong association rules.

**Business action:**  
Treat coffee availability and service speed as operational priorities. Position high-value complementary products near the coffee purchase flow.

**Important:**  
High co-purchase frequency does not prove that a discount is necessary. If customers already buy a combination naturally, discounting it may simply reduce margin.

---

### 3. Association rules can drive targeted bundles

Rules with both meaningful support and lift above 1 are better candidates for cross-selling than simply pairing the two most popular items.

**Business action:**  
Use rules such as `Toast → Coffee` or other high-lift combinations to design:

- POS recommendations
- menu-board suggestions
- digital ordering prompts
- limited bundle experiments

Measure incremental conversion before making them permanent.

---

### 4. Staffing and production should follow hourly demand

Transaction volume is concentrated in specific hours.

**Business action:**  

- schedule more front-of-house capacity around the observed peak;
- prepare fast-moving products shortly before demand rises;
- use quieter periods for restocking, cleaning and prep work.

This is operational optimization based on transaction flow, not just total sales.

---

### 5. Day-of-week demand should influence planning

Average daily transaction volume differs by weekday.

**Business action:**  
Create weekday-specific production and staffing templates instead of using the same plan every day.

Low-demand days are also useful for testing promotions without disrupting peak operations.

---

### 6. The product long tail deserves an assortment review

Many SKUs individually account for less than 1% of item volume.

**Business action:**  
Do **not** automatically remove them. Combine sales frequency with:

- gross margin;
- ingredient overlap;
- preparation time;
- shelf life;
- food waste;
- strategic/menu value.

Products that are low-volume, low-margin and waste-intensive become strong rationalization candidates.

---

## What this dataset cannot tell us

Without price, margin, stock and waste data, this analysis cannot legitimately estimate:

- profit uplift;
- optimal pricing;
- waste reduction in euros;
- inventory reorder quantities;
- promotion ROI.

Those would require additional business data.

This limitation is useful in a professional portfolio: good Data Science distinguishes evidence from assumptions.

## 10. Recommended next iteration

For a real bakery client, the next version of this analysis should add:

1. Unit price and cost.
2. Product margin.
3. Waste / unsold quantity.
4. Inventory and stockouts.
5. Promotion history.
6. Weather and local events.
7. Customer identifier or loyalty data.

That would allow the project to evolve from **descriptive + association analytics** into:

**Demand forecasting → production planning → waste optimization → profitability optimization**